# CBMLoss: Concept Bottleneck Models com Mitigação de Concept Leakage
Este notebook está configurado para clonar o repositório no ambiente do Google Colab e persistir todos os **checkpoints, métricas e dados no seu Google Drive** (`/content/drive/MyDrive/CBMLoss_Checkpoints`).

**Como funciona:**
- O repositório é clonado em `/content/CBMLoss`;
- A pasta `checkpoints/` é vinculada diretamente à sua pasta no Google Drive (`CBMLoss_Checkpoints`);
- Você pode colar seus checkpoints antigos manualmente nessa pasta do Drive;
- Todos os novos treinamentos e arquivos gerados (como `leakage_direct_metrics.csv`) ficam salvos permanentemente no seu Google Drive.

In [ ]:
# =============================================================================
# 1. MONTAR GOOGLE DRIVE E CONFIGURAR PASTA PERMANENTE DE CHECKPOINTS
# =============================================================================
import os
from google.colab import drive

# 1. Monta o Google Drive
drive.mount('/content/drive')

# 2. Cria a pasta permanente para checkpoints e métricas no Google Drive
GDRIVE_CHECKPOINTS = '/content/drive/MyDrive/CBMLoss_Checkpoints'
os.makedirs(GDRIVE_CHECKPOINTS, exist_ok=True)

print(f'>>> Google Drive montado com sucesso!')
print(f'>>> Pasta persistente de checkpoints: {GDRIVE_CHECKPOINTS}')
print(f'>>> Arquivos atualmente na pasta: {os.listdir(GDRIVE_CHECKPOINTS)}')
print('>>> DICA: Você pode colar os seus checkpoints antigos (.pth) diretamente nessa pasta do Drive!')


In [ ]:
# =============================================================================
# 2. CLONAR OU ATUALIZAR O REPOSITÓRIO GITHUB E VINCULAR CHECKPOINTS
# =============================================================================
import os
import shutil

# 1. Clona ou atualiza o repositório
if not os.path.exists('/content/CBMLoss'):
    !git clone https://github.com/paulohbl/CBMLoss.git /content/CBMLoss
else:
    !cd /content/CBMLoss && git reset --hard && git pull

# 2. Navega para a pasta do projeto
%cd /content/CBMLoss

# 3. Vincula a pasta de checkpoints do repositório ao Google Drive via symlink
GDRIVE_CHECKPOINTS = '/content/drive/MyDrive/CBMLoss_Checkpoints'
if os.path.islink('/content/CBMLoss/checkpoints'):
    os.unlink('/content/CBMLoss/checkpoints')
elif os.path.exists('/content/CBMLoss/checkpoints'):
    for item in os.listdir('/content/CBMLoss/checkpoints'):
        src = os.path.join('/content/CBMLoss/checkpoints', item)
        dst = os.path.join(GDRIVE_CHECKPOINTS, item)
        if not os.path.exists(dst):
            shutil.copy2(src, dst)
    shutil.rmtree('/content/CBMLoss/checkpoints')

os.symlink(GDRIVE_CHECKPOINTS, '/content/CBMLoss/checkpoints')
print(f'>>> Diretório de trabalho atual: {os.getcwd()}')
print(f'>>> Pasta checkpoints vinculada ao Google Drive: {os.readlink("checkpoints")}')


In [ ]:
# =============================================================================
# 3. INSTALAR DEPENDÊNCIAS DO PROJETO
# =============================================================================
!pip install -r requirements.txt -q
print('>>> Dependências instaladas com sucesso!')


In [ ]:
# =============================================================================
# 4. DOWNLOAD E PREPARAÇÃO DO DATASET CUB-200-2011
# =============================================================================
# Baixa o CUB-200 (~1.1 GB com barra de progresso) e processa os atributos.
# Se você já colocou 'CUB_200_2011.tgz' no seu Google Drive (CBMLoss_Checkpoints),
# ele copia direto do Drive sem precisar baixar da web!
!python download_datasets.py


## FASE 2: Estudos de Ablação Desacoplada e Medição Direta de Leakage
Os experimentos abaixo respondem diretamente aos pedidos dos revisores do SIBGRAPI:
1. **Baseline sem Regularização:** $\lambda_{ent}=0.0, \lambda_{ortho}=0.0$
2. **Isolamento da Entropia:** $\lambda_{ent}=0.5, \lambda_{ortho}=0.0$
3. **Isolamento da Descorrelação:** $\lambda_{ent}=0.0, \lambda_{ortho}=0.5$
4. **Configurações Leves Isoladas:** $0.1 / 0.0$ e $0.0 / 0.1$
5. **Medição Direta de Leakage:** Gap contínuo-discreto e Sonda Linear sobre o ruído residual dos conceitos.

*Todos os novos checkpoints e arquivos CSV são salvos automaticamente no seu Google Drive em `CBMLoss_Checkpoints/`.*

In [ ]:
# Baseline: Modelo CBM sem regularização (lambda_ent=0.0, lambda_ortho=0.0)
!python main.py --dataset cub200 --epochs 100 --lambda_ent 0.0 --lambda_ortho 0.0 --pretrained --checkpoint_dir checkpoints --patience 5


In [ ]:
# Ablação 1: Apenas Entropia (lambda_ent=0.5, lambda_ortho=0.0)
!python main.py --dataset cub200 --epochs 100 --lambda_ent 0.5 --lambda_ortho 0.0 --pretrained --checkpoint_dir checkpoints --patience 5


In [ ]:
# Ablação 2: Apenas Descorrelação (lambda_ent=0.0, lambda_ortho=0.5)
!python main.py --dataset cub200 --epochs 100 --lambda_ent 0.0 --lambda_ortho 0.5 --pretrained --checkpoint_dir checkpoints --patience 5


In [ ]:
# Ablação 3 (Leve): Apenas Entropia (lambda_ent=0.1, lambda_ortho=0.0)
!python main.py --dataset cub200 --epochs 100 --lambda_ent 0.1 --lambda_ortho 0.0 --pretrained --checkpoint_dir checkpoints --patience 5


In [ ]:
# Ablação 4 (Leve): Apenas Descorrelação (lambda_ent=0.0, lambda_ortho=0.1)
!python main.py --dataset cub200 --epochs 100 --lambda_ent 0.0 --lambda_ortho 0.1 --pretrained --checkpoint_dir checkpoints --patience 5


### Medição Direta de Concept Leakage
Executa a medição direta (Linear Probing sobre os resíduos $\boldsymbol{r} = \hat{\boldsymbol{c}} - \boldsymbol{c}$ e Discretization Gap $\Delta_{disc} = \text{Acc}_{soft} - \text{Acc}_{hard}$) em todos os modelos salvos em `checkpoints/`. Salva `leakage_direct_metrics.csv` diretamente no seu Google Drive.

In [ ]:
# Medição Direta de Leakage em todos os checkpoints disponíveis no Google Drive
!python measure_leakage.py --dataset cub200 --checkpoint_dir checkpoints --output_csv checkpoints/leakage_direct_metrics.csv


## Histórico: Execuções Anteriores (Ablação Conjunta $\lambda_{ent} = \lambda_{ortho}$)
Células abaixo mantidas como referência dos modelos treinados na fase 1.

In [ ]:
# Treino original: lambda=0.0
# !python main.py --dataset cub200 --epochs 100 --lambda_ent 0.0 --lambda_ortho 0.0 --pretrained --checkpoint_dir checkpoints --patience 5


In [ ]:
# Treino original: lambda=0.1
# !python main.py --dataset cub200 --epochs 100 --lambda_ent 0.1 --lambda_ortho 0.1 --pretrained --checkpoint_dir checkpoints --patience 5


In [ ]:
# Treino original: lambda=0.3
# !python main.py --dataset cub200 --epochs 100 --lambda_ent 0.3 --lambda_ortho 0.3 --pretrained --checkpoint_dir checkpoints --patience 5


In [ ]:
# Treino original: lambda=0.5
# !python main.py --dataset cub200 --epochs 100 --lambda_ent 0.5 --lambda_ortho 0.5 --pretrained --checkpoint_dir checkpoints --patience 5


In [ ]:
# Treino original: lambda=0.7
# !python main.py --dataset cub200 --epochs 100 --lambda_ent 0.7 --lambda_ortho 0.7 --pretrained --checkpoint_dir checkpoints --patience 5
